In [1]:
from pytorch_mlp_framework.model_architectures import ConvolutionalDimensionalityReductionBlockBN
from pytorch_mlp_framework.model_architectures import ConvolutionalProcessingBlockBN
from pytorch_mlp_framework.model_architectures import ConvolutionalProcessingBlockResBN
import torch
import torch.nn as nn
import numpy as np
import unittest
from unittest import TestCase

In [2]:
def default_conv_block_bn():
    input_shape = (10, 16, 32, 32)  # Batch size=10, filters=16, H=W=32
    num_filters = 16
    kernel_size = 3
    padding = 1
    bias = False
    dilation = 1
    return ConvolutionalProcessingBlockBN(
        input_shape=input_shape,
        num_filters=num_filters,
        kernel_size=kernel_size,
        padding=padding,
        bias=bias,
        dilation=dilation,
    )

In [3]:
def default_cnn_reduction_block_bn():
    input_shape = (10, 16, 32, 32)  # Batch size=10, filters=16, H=W=32
    num_filters = 16
    kernel_size = 3
    padding = 1
    bias = False
    dilation = 1
    reduction_factor = 2
    return ConvolutionalDimensionalityReductionBlockBN(
        input_shape=input_shape,
        num_filters=num_filters,
        kernel_size=kernel_size,
        padding=padding,
        bias=bias,
        dilation=dilation,
        reduction_factor=reduction_factor
    )

In [4]:
def default_conv_block_res_bn():
    input_shape = (10, 16, 32, 32)  # Batch size=10, filters=16, H=W=32
    num_filters = 16
    kernel_size = 3
    padding = 1
    bias = False
    dilation = 1
    return ConvolutionalProcessingBlockResBN(
        input_shape=input_shape,
        num_filters=num_filters,
        kernel_size=kernel_size,
        padding=padding,
        bias=bias,
        dilation=dilation,
    )

In [5]:
class TestCNNBlocks(unittest.TestCase):
    def test_output_shape_cnn_bn(self):
        # Initialize the blocks
        conv_block_bn = default_conv_block_bn()
        reduction_block_bn = default_cnn_reduction_block_bn()
        conv_block_res_bn = default_conv_block_res_bn()

        # Create input tensors
        input_tensor_conv_block = torch.rand(conv_block_bn.input_shape)
        input_tensor_reduction_block = torch.rand(reduction_block_bn.input_shape)
        input_tensor_conv_block_res = torch.rand(conv_block_res_bn.input_shape)

        # Forward pass through the reduction block
        output_tensor_reduction_block = reduction_block_bn.forward(input_tensor_reduction_block)
        self.assertEqual(
            output_tensor_reduction_block.shape,
            (10, reduction_block_bn.num_filters, 16, 16),
            "Output shape mismatch for ConvolutionalDimensionalityReductionBlockBN",
        )

        # Forward pass through the conv block
        output_tensor_conv_block = conv_block_bn.forward(input_tensor_conv_block)
        self.assertEqual(
            output_tensor_conv_block.shape,
            (10, conv_block_bn.num_filters, 32, 32),
            "Output shape mismatch for ConvolutionalProcessingBlockBN",
        )
        
        # Forward pass through the conv block with residual connections
        output_tensor_conv_block_res = conv_block_bn.forward(input_tensor_conv_block_res)
        self.assertEqual(
            output_tensor_conv_block_res.shape,
            (10, conv_block_res_bn.num_filters, 32, 32),
            "Output shape mismatch for ConvolutionalProcessingBlockResBN",
        )

    def test_contains_batch_norm(self):
        conv_block_bn = default_conv_block_bn()
        reduction_block_bn = default_cnn_reduction_block_bn()
        conv_block_res_bn = default_conv_block_res_bn()
        
        self.assertTrue(
            any(isinstance(layer, torch.nn.BatchNorm2d) for layer in reduction_block_bn.layer_dict.values()),
            "Batch normalization layer not found in ConvolutionalDimensionalityReductionBlockBN",
        )
        
        self.assertTrue(
            any(isinstance(layer, torch.nn.BatchNorm2d) for layer in conv_block_bn.layer_dict.values()),
            "Batch normalization layer not found in ConvolutionalProcessingBlockBN",
        )
        
        self.assertTrue(
            any(isinstance(layer, torch.nn.BatchNorm2d) for layer in conv_block_res_bn.layer_dict.values()),
            "Batch normalization layer not found in ConvolutionalProcessingBlockResBN",
        )
        
    def test_fprop(self):
        conv_block_bn = default_conv_block_bn()
        reduction_block_bn = default_cnn_reduction_block_bn()
        conv_block_res_bn = default_conv_block_res_bn()
        
        # Create input tensors
        input_tensor_conv_block = torch.rand(conv_block_bn.input_shape)
        input_tensor_reduction_block = torch.rand(reduction_block_bn.input_shape)
        input_tensor_conv_block_res = torch.rand(conv_block_res_bn.input_shape)
        
        # Forward propagate
        try:
            reduction_output_tensor = reduction_block_bn(input_tensor_reduction_block)
            print("Output shape:", reduction_output_tensor.shape)  # Debugging output
        except Exception as e:
            self.fail(f"Forward pass raised an exception: {e}")
            
        # Forward propagate
        try:
            conv_output_tensor = conv_block_bn(input_tensor_conv_block)
            print("Output shape:", conv_output_tensor.shape)  # Debugging output
        except Exception as e:
            self.fail(f"Forward pass raised an exception: {e}")
        
        try:
            res_conv_output_tensor = conv_block_res_bn(input_tensor_conv_block_res)
            print("Output shape:", res_conv_output_tensor.shape)  # Debugging output
        except Exception as e:
            self.fail(f"Forward pass raised an exception: {e}")

In [6]:
suite = unittest.TestLoader().loadTestsFromTestCase(TestCNNBlocks)
unittest.TextTestRunner().run(suite)

...
----------------------------------------------------------------------
Ran 3 tests in 0.069s

OK


torch.Size([10, 16, 32, 32])
torch.Size([10, 16, 16, 16])
torch.Size([10, 16, 32, 32])
torch.Size([10, 16, 32, 32])
torch.Size([10, 16, 16, 16])
torch.Size([10, 16, 32, 32])
Output shape: torch.Size([10, 16, 16, 16])
Output shape: torch.Size([10, 16, 32, 32])
Output shape: torch.Size([10, 16, 32, 32])
torch.Size([10, 16, 32, 32])
torch.Size([10, 16, 16, 16])
torch.Size([10, 16, 32, 32])


<unittest.runner.TextTestResult run=3 errors=0 failures=0>